# Strategy Analysis
- Database: PostreSQL
- Dataframe: Polars
- Stock: Single stock (AAPL)
- Time frame: 2 years
- Uses high, low, close
- MACD, AROON, STOCH
- Candlestick patterns

In [55]:
# Perform imports
import polars as pl
import polars_talib as plta
from datetime import date, datetime
import altair as alt

from securities_load.securities.polar_table_functions import (retrieve_ohlcv_using_ticker_ids_and_dates)

In [56]:
# Define variables
symbols = [5219,5230]
today = datetime.now()
min_date = date(today.year - 1, today.month, today.day)  # Two years ago from today
start = min_date.strftime("%Y-%m-%d")
end = today.strftime("%Y-%m-%d")
smas = (50,200)
emas = (12, 26)
filter_on = "AAPL"

In [57]:
# Get the close data for all symbols
pl_df = retrieve_ohlcv_using_ticker_ids_and_dates(symbols, start, end)
pl_df.head()

id,ticker,date,open,high,low,close,volume
i32,str,date,f64,f64,f64,f64,i64
5219,"""AACT""",2024-08-12,10.74,10.76,10.74,10.76,20508
5219,"""AACT""",2024-08-13,10.75,10.76,10.75,10.755,5131
5219,"""AACT""",2024-08-14,10.76,10.76,10.75,10.755,76000
5219,"""AACT""",2024-08-15,10.751,10.77,10.75,10.77,415336
5219,"""AACT""",2024-08-16,10.77,10.77,10.76,10.76,33306


In [58]:
# Extract some basic information
last_date = pl_df[-1, "date"]
last_close = pl_df[-1, "close"]
last_open = pl_df[-1, "open"]
last_high = pl_df[-1, "high"]
last_low = pl_df[-1, "low"]
last_volume = pl_df[-1, "volume"]
change = pl_df[-1, "close"] - pl_df[-2, "close"]
percent_change = (change / pl_df[-2, "close"]) * 100

In [ ]:
# Get some technical indicators
pl_df = pl_df.with_columns(
    pl.col("date").dt.strftime("%Y-%m-%d").alias("date_as_string"),
    *[plta.sma(pl.col("close"),timeperiod=i).over("ticker").alias(f"sma{i:03d}") for i in smas],
    *[plta.ema(pl.col("close"),timeperiod=i).over("ticker").alias(f"ema{i:03d}") for i in emas],
    plta.macd(fastperiod=10, slowperiod=20,signalperiod=5).over("ticker").alias("macd"),
    plta.rsi().over("ticker").alias("rsi"),
    plta.ht_dcperiod().over("ticker").alias("ht_dcp"),
    plta.aroon().over("ticker").alias("aroon"),
    plta.wclprice().over("ticker").alias("wclprice"),
    plta.stoch(pl.col("high"), pl.col("low"), pl.col("close"), fastk_period=14, slowk_period=7, slowd_period=7).over("ticker").alias("stoch"),
    plta.cdl2crows().over("ticker").alias("cdl2crows"),
    plta.cdlhammer().over("ticker").alias("cdlhammer"),
).with_columns(
    pl.col("macd").struct.field("macd"),
    pl.col("macd").struct.field("macdsignal"),
    pl.col("macd").struct.field("macdhist"),
    pl.col("aroon").struct.field("aroondown"),
    pl.col("aroon").struct.field("aroonup"),
    pl.col("stoch").struct.field("slowk"),
    pl.col("stoch").struct.field("slowd"),
).select(
    pl.exclude(["macd", "aroon", "stoch"])
).filter(
    pl.col("ticker") == filter_on
)

In [60]:
# Define y-values for the position of the mark for the hammers
pl_df = pl_df.with_columns(
    pl.when(pl.col("cdlhammer") == 100.0).then(pl.col("wclprice") * 1.025
        ).when(pl.col("cdlhammer") == -100.0).then(pl.col("wclprice") * 0.975
        ).otherwise(None).alias("hammer")
)
pl_df.head()

id,ticker,date,open,high,low,close,volume,date_as_string,sma050,sma200,ema012,ema026,rsi,ht_dcp,wclprice,cdl2crows,cdlhammer,macdsignal,macdhist,aroondown,aroonup,slowk,slowd,hammer
i32,str,date,f64,f64,f64,f64,i64,str,f64,f64,f64,f64,f64,f64,f64,i32,i32,f64,f64,f64,f64,f64,f64,f64
5230,"""AAPL""",2024-08-12,216.07,219.5099,215.6,217.53,38028092,"""2024-08-12""",NaN,NaN,NaN,NaN,NaN,NaN,217.542475,0,0,NaN,NaN,NaN,NaN,NaN,NaN,null
5230,"""AAPL""",2024-08-13,219.01,221.89,219.01,221.27,44155331,"""2024-08-13""",NaN,NaN,NaN,NaN,NaN,NaN,220.86,0,0,NaN,NaN,NaN,NaN,NaN,NaN,null
5230,"""AAPL""",2024-08-14,220.57,223.03,219.7,221.72,41948574,"""2024-08-14""",NaN,NaN,NaN,NaN,NaN,NaN,221.5425,0,0,NaN,NaN,NaN,NaN,NaN,NaN,null
5230,"""AAPL""",2024-08-15,224.6,225.35,222.76,224.72,46398713,"""2024-08-15""",NaN,NaN,NaN,NaN,NaN,NaN,224.3875,0,0,NaN,NaN,NaN,NaN,NaN,NaN,null
5230,"""AAPL""",2024-08-16,223.92,226.8271,223.6501,226.05,44338440,"""2024-08-16""",NaN,NaN,NaN,NaN,NaN,NaN,225.6443,0,0,NaN,NaN,NaN,NaN,NaN,NaN,null


In [65]:
charts = []

MA_colors = ["blue", "orange", "violet"]
base = alt.Chart(pl_df).encode(
    x=alt.X(
        "date:T",
        title="",
        # timeUnit="yearmonthdate",
        axis=alt.Axis(
            tickColor="#000000",
            domainColor="#D0D3D3",
            gridColor="#D0D3D3",
            labelColor="#000000",
            titleColor="#000000",
            labelFontSize=12,
            titleFontSize=16,
            grid=False,
            labelAngle=-45,
            labels=False,
        ),
    ),
)
# Shadows
rule = base.mark_rule().encode(
    y=alt.Y(
        "low:Q",
        title="Price",
        axis=alt.Axis(
            tickColor="#000000",
            domainColor="#D0D3D3",
            gridColor="#D0D3D3",
            labelColor="#000000",
            titleColor="#000000",
            labelFontSize=12,
            titleFontSize=16,
            grid=True,
            labels=True,
        ),
        scale=alt.Scale(zero=False),
    ),
    y2=alt.Y2("high:Q"),
    color=alt.condition(
        "datum.open <= datum.close", alt.value("#23DE0E"), alt.value("#f6102b")
    ),
)
# Real body
bar = (
    base.mark_bar(size = 3)
    .encode(
        y=alt.Y("open:Q"),
        y2=alt.Y2("close:Q"),
        color=alt.condition(
            "datum.open <= datum.close", alt.value("#23DE0E"), alt.value("#f6102b")
        ),
        tooltip=["date:T", "open", "high", "low", "close"],
    )
    .properties(
        width=1300,
        height=400,
        title={
            "text": f"Candlestick for {filter_on}",
            "subtitle": [
                f"Date: {last_date}, Close: {last_close:.2f}, Change: {change:.2f} ({percent_change:.2f}%) Volume: {last_volume}",
            ],
            "fontSize": 18,
            "subtitleFontSize": 14,
        },
    )
)

candlestick = rule + bar

# Simple Moving Average
color_count = 0
for i in smas:
    ave_name = f"sma{i:03d}"
    sma = base.mark_line(color=MA_colors[color_count]).encode(
        y=alt.Y(ave_name + ":Q"),
        tooltip=["date", ave_name + ":Q"],
    )
    color_count += 1
    candlestick = candlestick + sma

# Exponential Moving Average
color_count = 0
for i in emas:
    ema_name = f"ema{i:03d}"
    ema = base.mark_line(color=MA_colors[color_count], strokeDash=[5, 5]).encode(
        y=alt.Y(ema_name + ":Q"),
        tooltip=["date", ema_name + ":Q"],
    )
    color_count += 1
    candlestick = candlestick + ema

hammer = base.mark_point(shape="arrow", angle=180, size=100).encode(
    y=alt.Y("hammer:Q"), color=alt.value("#000000")
)
candlestick = candlestick + hammer

hammer_text = base.mark_text(align="left", baseline="bottom", text="Hammer").encode(
    y=alt.Y("hammer:Q"),
)
candlestick = candlestick + hammer_text

charts.append(candlestick)

volume_base = alt.Chart(pl_df).encode(
    x=alt.X(
        "date:T",
        title="",
        axis=alt.Axis(
            tickColor="#000000",
            domainColor="#D0D3D3",
            gridColor="#D0D3D3",
            labelColor="#000000",
            titleColor="#000000",
            labelFontSize=12,
            titleFontSize=16,
            grid=False,
            labelAngle=-45,
            labels=True,
        ),
    ),
)
volume = (
    volume_base.mark_bar(size=3, color="dodgerblue")
    .encode(
        y=alt.Y(
            "volume:Q",
            title="Volume",
            axis=alt.Axis(
                tickColor="#000000",
                domainColor="#D0D3D3",
                gridColor="#D0D3D3",
                labelColor="#000000",
                titleColor="#000000",
                labelFontSize=12,
                titleFontSize=16,
                grid=True,
                labels=True,
            ),
        ),
        tooltip=["date:T", "volume:Q"],
    )
    .properties(width=1300, height=200)
)


# color_count = 0
# for i in vmas:
#     vma_name = f"vma{i:03d}"
#     vma = base.mark_line(color=MA_colors[color_count]).encode(
#         y=alt.Y(vma_name + ":Q"),
#         tooltip=["date", vma_name + ":Q"],
#     )
#     color_count += 1
#     volume = volume + vma


macd =  alt.Chart(pl_df
    ).encode(
        x=alt.X(
            "date:T",
            title="",
            axis=alt.Axis(
                tickColor="#000000",
                domainColor="#D0D3D3",
                gridColor="#D0D3D3",
                labelColor="#000000",
                titleColor="#000000",
                labelFontSize=12,
                titleFontSize=16,
                grid=False,
                labelAngle=-45,
                labels=False,
            ),
        )
    ).properties(width=1300, height=100, title="")

macd_line = macd.mark_line(color="dodgerblue", strokeWidth=1).encode(
    y=alt.Y(
        "macd:Q",
        title="MACD & Signal",
        axis=alt.Axis(
            tickColor="#000000",
            domainColor="#D0D3D3",
            gridColor="#D0D3D3",
            labelColor="#000000",
            titleColor="#000000",
            labelFontSize=12,
            titleFontSize=16,
            grid=False,
            labels=True,
        ),
    ),
    tooltip=["date:T", "macd:Q"],
)

signal_line = macd.mark_line(color="darkorange", strokeWidth=1).encode(
    y=alt.Y("macdsignal:Q", title=""), tooltip=["date:T", "macdsignal:Q"]
)

macd_lines = (
    alt.layer(
        signal_line,
        macd_line,
    )
    .resolve_scale(x="shared", y="shared")
    .properties(title="", width=1300, height=100)
)

histogram = macd.mark_bar(size=3).encode(
    y=alt.Y(
        "macdhist:Q",
        title="MACD Hist",
        axis=alt.Axis(
            # format="f",
            tickColor="#9D9BA1",
            domainColor="#000000",
            gridColor="#D0D3D3",
            labelColor="#000000",
            titleColor="#000000",
            grid=True,
            labelFontSize=12,
            titleFontSize=16,
        ),
    ),
    color=alt.condition(
        alt.datum.macdhist > 0,  # Positive bars
        alt.value("green"),
        alt.value("red"),  # Negative bars
    ),
    tooltip=["date:T", "macdhist:Q", "macd:Q", "macdsignal:Q"],
)

macd_chart = (
    alt.layer(
        histogram,
        macd_lines,
    )
    .resolve_scale(x="shared", y="independent")
    .properties(title="", width=1300, height=100)
)
charts.append(macd_chart)


# Relative Strength Index (RSI)
# Create a base chart for RSI
rsi =  alt.Chart(pl_df
).encode(
    x=alt.X(
        "date:T",
        title="",
        axis=alt.Axis(
            tickColor="#D0D3D3",
            domainColor="#000000",
            gridColor="#D0D3D3",
            labelColor="#000000",
            titleColor="#000000",
            labelFontSize=12,
            titleFontSize=16,
            grid=False,
            labelAngle=-45,
            labels=False,
        ),
    )
).properties(width=1300, height=100, title="")


rsi_line = rsi.mark_line(color="dodgerblue", strokeWidth=1).encode(
    y=alt.Y(
        "rsi:Q",
        title="RSI",
        axis=alt.Axis(
            tickColor="#D0D3D3",
            domainColor="#000000",
            gridColor="#D0D3D3",
            labelColor="#000000",
            titleColor="#000000",
            labelFontSize=12,
            titleFontSize=16,
            grid=True,
            labels=True,
        ),
    ),
    tooltip=["date:T", "rsi:Q"],
)
yrule_upper = (
    alt.Chart()
    .mark_rule(strokeDash=[2, 2], color="#9219E9")
    .encode(y=alt.datum(70))
)

yrule_lower = (
    alt.Chart()
    .mark_rule(strokeDash=[2, 2], color="#9219E9")
    .encode(y=alt.datum(30))
)

rsi_fill = rsi.mark_area(color="#9219E9", opacity=0.1).encode(
    y=alt.datum(30), y2=alt.datum(70)
)

rsi_chart = rsi_line + yrule_lower + yrule_upper + rsi_fill
charts.append(rsi_chart)

charts.append(volume)

combined_chart = (
    alt.vconcat(*charts).resolve_scale(x="shared").configure(background="#eaf3fb")
)

combined_chart


alt.VConcatChart(...)